# Exploratory Data Analysis: Personal Budget Anomaly Detector

**Objective:** Before applying machine learning algorithms like Isolation Forest to detect unusual spending, we first need to understand the shape, distribution, and general patterns within our transaction data. 

In this notebook, we will:
1. Load the raw transaction data.
2. Check for missing values and data types.
3. Visualize spending by category and over time.
4. Prototype the time-based features (Day of Week, Weekend) needed for our AI model.

In [ ]:
# Import necessary libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style for plots
sns.set_theme(style="whitegrid")

## 1. Loading and Inspecting the Data
Let's load our raw bank statement (the dummy dataset) and take a look at the first few rows to understand its structure.

In [ ]:
# Load the data
df = pd.read_csv('../data/raw_transactions_dummy.csv')

# Display the first 5 rows
display(df.head())

# Check data types and look for missing (null) values
print("\n--- Data Info ---")
df.info()

*Observation:* The `Date` column is currently being read as a standard string (object). We need to convert this into a proper datetime format so we can extract useful features from it later. There are no missing values, which is great.

## 2. Statistical Summary
Let's look at the basic math behind our spending amounts to establish a baseline.

In [ ]:
# Get basic statistics for the numerical columns (Amount)
display(df.describe())

*Observation:* The mean (average) transaction is quite heavily skewed by a few large purchases (like the mess fee). The 75th percentile is much lower than the maximum, which is a strong hint that anomalies/outliers exist in this dataset.

## 3. Visualizing Spending Patterns
To help our anomaly detector, let's see where the money is going by category. If we only buy "Software" once a year, a new software charge is highly anomalous.

In [ ]:
# Plot total spending by category
plt.figure(figsize=(10, 5))
category_spending = df.groupby('Category')['Amount'].sum().sort_values(ascending=False)

sns.barplot(x=category_spending.values, y=category_spending.index, palette="viridis")
plt.title('Total Spending by Category')
plt.xlabel('Total Amount (INR)')
plt.ylabel('Category')
plt.show()

Next, let's look at a timeline of all transactions to see if any obvious spikes stand out visually before we even apply the AI.

In [ ]:
# Convert Date to datetime for plotting
df['Date'] = pd.to_datetime(df['Date'])

plt.figure(figsize=(12, 5))
sns.scatterplot(x='Date', y='Amount', hue='Category', data=df, s=100, alpha=0.8)
plt.title('Transaction Amounts Over Time')
plt.xlabel('Date')
plt.ylabel('Amount (INR)')
plt.xticks(rotation=45)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

*Observation:* We can clearly see a few massive spikes (e.g., the 4500 INR Mess Fee and the 3299 INR Entertainment purchase). However, to build a *smart* model, we can't just flag high amounts. A 150 INR food charge at 3 AM on a Tuesday might be more anomalous than a 4500 INR tuition fee on the 1st of the month.

## 4. Feature Engineering Prototype
To give the Isolation Forest model context, we will engineer new columns from the date: `Day_of_Week` and `Is_Weekend`.

In [ ]:
# Extract day of the week (0 = Monday, 6 = Sunday)
df['Day_of_Week'] = df['Date'].dt.dayofweek

# Create a boolean column for weekends (Saturday=5, Sunday=6)
df['Is_Weekend'] = df['Day_of_Week'].apply(lambda x: 1 if x >= 5 else 0)

# Show the new features alongside the original data
display(df[['Date', 'Description', 'Amount', 'Day_of_Week', 'Is_Weekend']].head(10))

## Conclusion
The data is clean and ready. By engineering the `Day_of_Week` and `Is_Weekend` features, we have provided the necessary context for our machine learning model. 

The next step is to formalize this cleaning process into `src/data_cleaning.py` and feed the structured data into our Isolation Forest algorithm in `src/anomaly_model.py`.